# Seq2Seq Practice Notebook (Translation, Summarization, QA)

This notebook practices sequence-to-sequence modeling with Hugging Face Transformers.

Tasks covered:
- Machine translation (English -> French)
- Abstractive summarization
- Text-to-text question answering / instruction following

## 1) Install dependencies (run once)

In [ ]:
%pip install -q transformers datasets sentencepiece sacremoses torch evaluate

## 2) Load helper imports

In [ ]:
import torch
from datasets import load_dataset
from transformers import pipeline

In [ ]:
device = 0 if torch.cuda.is_available() else -1
print("GPU available:", torch.cuda.is_available())

## 3) Seq2Seq task: Machine Translation

In [ ]:
translator = pipeline(
    task="translation_en_to_fr",
    model="Helsinki-NLP/opus-mt-en-fr",
    device=device,
)

sentences = [
    "The company reported strong revenue growth this quarter.",
    "Investors are watching inflation and interest rates closely.",
]

translations = translator(sentences, max_length=80)
for src, out in zip(sentences, translations):
    print("EN:", src)
    print("FR:", out["translation_text"])
    print()

## 4) Seq2Seq task: Abstractive Summarization

In [ ]:
# Load a small sample from a summarization dataset
cnn = load_dataset("cnn_dailymail", "3.0.0", split="test[:3]")
cnn

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

sum_model_id = "facebook/bart-large-cnn"
sum_tokenizer = AutoTokenizer.from_pretrained(sum_model_id)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(sum_model_id)
sum_model = sum_model.to("cuda") if torch.cuda.is_available() else sum_model

def summarize_text(text, max_new_tokens=130, min_new_tokens=40):
    inputs = sum_tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = sum_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            do_sample=False,
        )
    return sum_tokenizer.decode(output_ids[0], skip_special_tokens=True)

article = cnn[0]["article"]
reference = cnn[0]["highlights"]
pred = summarize_text(article, max_new_tokens=130, min_new_tokens=40)

print("ARTICLE (first 700 chars):\n", article[:700], "...\n")
print("MODEL SUMMARY:\n", pred, "\n")
print("REFERENCE SUMMARY:\n", reference)

## 5) Evaluate summarization quickly (ROUGE on tiny sample)

In [ ]:
import evaluate

rouge = evaluate.load("rouge")

predictions = []
references = []
for row in cnn:
    summary = summarize_text(row["article"], max_new_tokens=130, min_new_tokens=40)
    predictions.append(summary)
    references.append(row["highlights"])

scores = rouge.compute(predictions=predictions, references=references)
scores

## 6) Seq2Seq text-to-text QA / instruction following (T5/FLAN style)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

qa_model_id = "google/flan-t5-base"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_id)
qa_model = AutoModelForSeq2SeqLM.from_pretrained(qa_model_id)
qa_model = qa_model.to("cuda") if torch.cuda.is_available() else qa_model

context = (
    "Tesla reported a rise in quarterly deliveries, but margins were pressured by price cuts. "
    "Management said efficiency gains and software revenue are expected to support profitability."
)
question = "What pressured margins?"
prompt = f"Answer the question based on the context. Context: {context} Question: {question}"

inputs = qa_tokenizer(prompt, return_tensors="pt", truncation=True)
if torch.cuda.is_available():
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

with torch.no_grad():
    output_ids = qa_model.generate(**inputs, max_new_tokens=40, do_sample=False)

answer = qa_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("Q:", question)
print("A:", answer)

## 7) (Optional) Mini fine-tuning scaffold for translation with T5

Use this as a starting point when you want to train your own Seq2Seq model.
This cell only prepares tokenized data to keep runtime short.

In [ ]:
from transformers import AutoTokenizer

wmt = load_dataset("wmt16", "ro-en", split="train[:200]")
tokenizer = AutoTokenizer.from_pretrained("t5-small")

def preprocess(example):
    src = "translate Romanian to English: " + example["translation"]["ro"]
    tgt = example["translation"]["en"]
    model_inputs = tokenizer(src, max_length=128, truncation=True)
    labels = tokenizer(text_target=tgt, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_wmt = wmt.map(preprocess, remove_columns=wmt.column_names)
tokenized_wmt

## 8) Practice exercises

1. Replace translation model with another language pair from OPUS-MT.
2. Compare `facebook/bart-large-cnn` vs `google/pegasus-xsum` for summarization quality and style.
3. Build a prompt template for financial QA and test consistency on multiple contexts.
4. Fine-tune `t5-small` on a small subset and compare zero-shot vs fine-tuned outputs.